In [1]:
from selenium import webdriver
from selenium.webdriver import Chrome, ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import  WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
import pandas as pd 
import time, threading, os, requests, ast,json
from datetime import datetime


def format_auction_date(date_str):
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    day = dt.strftime("%d").lstrip("0")

    if 4 <= int(day) <= 20 or 24 <= int(day) <= 30:
        suffix = "th"
    else:
        suffix = ["st", "nd", "rd"][int(day) % 10 - 1]

    return dt.strftime(f"%a {day}{suffix} %b")




def scarpe(path,date_format="%d-%m-%Y"):
    options = ChromeOptions()
    options.headless = True
    service = Service(ChromeDriverManager().install())
    driver = Chrome(service=service, options=options)
    driver.get(path)
    wait = WebDriverWait(driver, 10)
    driver.maximize_window()

    try:
        login = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/a[text()="Login"]'))
        )
        login.click()
    except:
        print("no login")

    try:
        user_name = WebDriverWait(driver, 2).until(
            EC.presence_of_element_located((By.ID, 'username'))
        )
        user_name.send_keys("fourbrotherstrading@icloud.com")
    except Exception as e:
        print("Username error", e)

    try:
        password = WebDriverWait(driver, 2).until(
            EC.presence_of_element_located((By.ID, 'password'))
        )
        password.send_keys("Muhssan7865@")
    except Exception as e:
        print("Password error", e)

    try:
        check = WebDriverWait(driver, 2).until(
            EC.presence_of_element_located((By.XPATH, './/button[text()="Sign in"]'))
        )
        check.click()
    except:
        print("Sign-in button not found")

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//a[normalize-space()="Portal"]'))
        )

        print("Login successful")

        time.sleep(1) 
        driver.get(path)

    except:
        print("❌ Login failed or timeout")
        
    try:
        auction_header = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.auction-header.row"))
        )

        auction_name = auction_header.find_element(By.TAG_NAME, "h1").text.strip()

        header_ul = auction_header.find_element(By.CSS_SELECTOR, "div.col-12.col-xl-8 ul.header-details.icon-list")
        li_items = header_ul.find_elements(By.TAG_NAME, "li")

        center = li_items[0].text.strip() if len(li_items) > 0 else ""

        time_text = li_items[1].text.strip() if len(li_items) > 1 else ""
        if " - " in time_text:
            time_part, date_part = [x.strip() for x in time_text.split(" - ")]
        else:
            time_part, date_part = time_text, ""

        data = {
            "auction_name": auction_name,
            "center": center,
            "date": date_part,
            "time": time_part
        }


        database_file = "database.json"
        if os.path.exists(database_file):
            with open(database_file, "r", encoding="utf-8") as f:
                try:
                    existing_data = json.load(f)
                    if not isinstance(existing_data, list):
                        existing_data = [existing_data]
                except:
                    existing_data = []
        else:
            existing_data = []

        existing_data.append(data)


        with open(database_file, "w", encoding="utf-8") as f:
            json.dump(existing_data, f, ensure_ascii=False, indent=4)

        print(f"✅ Auction data saved to {database_file}")

    except Exception as e:
        print(f"❌ Error scraping or saving data: {e}")

    car_count = 0

    try:
        if not os.path.exists("html"):
            os.makedirs("html")

        while True:  
            try:
                car_images = WebDriverWait(driver, 5).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "img.card-img-top"))
                )

                for idx, img in enumerate(car_images):
                    try:
                        parent_link = img.find_element(By.XPATH, "./ancestor::a")
                        car_url = parent_link.get_attribute("href")
                        if not car_url:
                            print("No link found for this car, skipping.")
                            continue

                        driver.execute_script("window.open(arguments[0], '_blank');", car_url)
                        driver.switch_to.window(driver.window_handles[-1])
                        time.sleep(1) 
                        try:
                            li_items = WebDriverWait(driver, 5).until(
                                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.detail-item"))
                            )

                            reg_number = ""
                            for li in li_items:
                                try:
                                    span_text = li.find_element(By.TAG_NAME, "span").text.strip()
                                    if span_text.lower() == "registration": 
                                        strong_el = li.find_element(By.TAG_NAME, "strong")
                                        reg_number = strong_el.text.strip().replace("/", "_").replace(" ", "_")
                                        break
                                except Exception:
                                    continue

                            if not reg_number:
                                reg_number = f"car_{car_count+1}"

                        except Exception:
                            reg_number = f"car_{car_count+1}"


                        filename = f"html/{reg_number}.html"
                        with open(filename, "w", encoding="utf-8") as f:
                            f.write(driver.page_source)
                        print(f"✔ Saved HTML: {filename}")
                        car_count += 1

                        driver.close()
                        driver.switch_to.window(driver.window_handles[0])
                        time.sleep(1)

                    except Exception as e:
                        print("Error processing car image:", e)
                        continue

                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, "li.page-item.page-item-arrow-next a")
                    next_href = next_btn.get_attribute("href")
                    if next_href:
                        print(f"➡ Moving to next page: {next_href}")
                        driver.get(next_href)
                        time.sleep(2)
                    else:
                        print("No more pages.")
                        break
                except Exception:
                    print("Pagination finished.")
                    break

            except Exception:
                print("No car images found on this page, stopping loop.")
                break

    except Exception as e:
        print("❌ Fatal error during car scraping:", e)

    # print(f"\n✅ Total cars processed: {car_count}")
    
    # driver.quit()

path = 'https://www.ebca.co.uk/auction/79'
date ="2026-01-20"
scarpe(path,date)


Login successful
✅ Auction data saved to database.json
✔ Saved HTML: html/GU15RBX.html
✔ Saved HTML: html/FE18KMO.html
✔ Saved HTML: html/GY64ESG.html
✔ Saved HTML: html/HY63YLM.html
✔ Saved HTML: html/FP64NBD.html
✔ Saved HTML: html/LD66RXP.html
✔ Saved HTML: html/FD64FYC.html
✔ Saved HTML: html/PN63UTT.html
✔ Saved HTML: html/WM17KXV.html
✔ Saved HTML: html/GX21YZK.html
✔ Saved HTML: html/SC65DNY.html
✔ Saved HTML: html/GF17WPU.html
✔ Saved HTML: html/WF17DJK.html
✔ Saved HTML: html/GU63OJR.html
✔ Saved HTML: html/GU17PKX.html
✔ Saved HTML: html/GY70RNF.html
✔ Saved HTML: html/LM66HHL.html
✔ Saved HTML: html/WX16VUP.html
✔ Saved HTML: html/EU13VWN.html
✔ Saved HTML: html/KS65XBP.html
✔ Saved HTML: html/HK14HHX.html
✔ Saved HTML: html/EN19NAE.html
✔ Saved HTML: html/GV70BVL.html
✔ Saved HTML: html/GM68XJV.html
✔ Saved HTML: html/EX68ZDA.html
✔ Saved HTML: html/EX69NWF.html
✔ Saved HTML: html/FR11JNK.html
✔ Saved HTML: html/WV68KJE.html
✔ Saved HTML: html/KJ18DZC.html
✔ Saved HTML: htm

In [2]:
import os,re,json
import csv
from bs4 import BeautifulSoup
from datetime import datetime
with open(r"D:\bots\headers.json", "r", encoding="utf-8") as f:
    header_map = json.load(f)
headers = [header_map[k] for k in sorted(header_map, key=int)]

with open("database.json","r",encoding="utf-8") as f:
    database = json.load(f)
if isinstance(database, list):
    auctionDetails = database[0] if database else {}

elif isinstance(database, dict):
    auction_list = database.get("auction", [])
    auctionDetails = auction_list[0] if auction_list else {}

else:
    auctionDetails = {}
def get_base_folder_info():
    folder_name = os.path.basename(os.getcwd())

    parts = folder_name.split("-")
    if not parts or not parts[0].isdigit():
        return None, None

    sheet_id = parts[0]
    name_parts = parts[1:]
    if name_parts and name_parts[0].isdigit():
        name_parts = name_parts[1:]

    auction_name = "-".join(name_parts).strip()


    return sheet_id, auction_name

def extract_details(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    result = {}

  
    details_ul = soup.find("ul", class_="details-list")
    if details_ul:
        li_items = details_ul.find_all("li", class_="detail-item")
        for li in li_items:
            key_el = li.find("span")
            value_el = li.find("strong")
            if key_el and value_el:
                key = key_el.get_text(strip=True)
                value = value_el.get_text(strip=True)
                result[key] = value


    features_p = soup.find("p", class_="mt-4")
    if features_p:
        features_text = features_p.get_text(" ", strip=True)
        result["Features"] = features_text

    return result

def extract_additional_info(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    result = {}

    details_ul = soup.find("ul", class_="tab-grid details-list")
    if details_ul:
        li_items = details_ul.find_all("li", class_="detail-item")
        for li in li_items:
            key_el = li.find("span")
            value_el = li.find("strong")
            if key_el and value_el:
                key = key_el.get_text(strip=True)
                value = value_el.get_text(strip=True)
                result[key] = value


    if result:
        return {"information": result}
    else:
        return {"information": {}}


def yearGeter(date_str):
    if not date_str:
        return ""

    date_str = str(date_str).strip().replace("\\", "")

    for fmt in ("%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d", "%Y/%m/%d"):
        try:
            dt = datetime.strptime(date_str, fmt)
            return str(dt.year)
        except ValueError:
            continue

    return ""

def extract_doors(variant_tag):
    if not variant_tag:
        return ""

    match = re.search(r"(\d+)\s*dr", variant_tag, re.IGNORECASE)
    if match:
        return match.group(1)

    return ""

def varientClean(variant):
    if not variant:
        return ""

    v = variant
    v = re.sub(r"^\s*\d+(\.\d+)?\s*", "", v)
    v = re.sub(r"\b\d+(\.\d+)?\s*cc\b", "", v, flags=re.IGNORECASE)
    v = re.sub(r"^\s*\d+(\.\d+)?i\b", "", v, flags=re.IGNORECASE)
    v = re.sub(r"\b\d+\s*dr\b", "", v, flags=re.IGNORECASE)
    v = re.sub(r"[\[\]]", "", v)
    v = re.sub(r"\s{2,}", " ", v)

    return v.strip()


def extract_manual_keys():
    folder = "html"
    output_file = "Ecba_data.csv"

    all_rows = []

    for file in os.listdir(folder):
        if file.endswith(".html"):
            file_path = os.path.join(folder, file)
            with open(file_path, "r", encoding="utf-8") as f:
                html_content = f.read()
                soup = BeautifulSoup(html_content, "html.parser")

            row = {}

            reg_el = soup.find("span", class_="pill-item pill-item-reg")
            reg = reg_el.get_text(strip=True) if reg_el else ""
            pattern = re.compile(r'^[A-Z]{1,3}[0-9]{1,3}[A-Z]{1,3}$', re.I)
            if not pattern.match(reg):
                print(f"❌ Not valid: {reg} → Deleting file {file}")
                os.remove(file_path)
                continue
            else:
                row[header_map['5']] = reg
                lot_el = soup.find("span", class_="pill-item pill-item-lot")
                row[header_map['9']] = lot_el.get_text(strip=True).replace("Lot", "").strip() if lot_el else ""

                title_tag = soup.find("h2", class_="title-h2")
                row[header_map['4']] = title_tag.get_text(strip=True) if title_tag else ""
                row[header_map['6']] = row[header_map['4']] 

                model_tag = soup.find("p", class_="title-sub")
                row[header_map['7']] = model_tag.get_text(strip=True) if model_tag else ""

                variant_tag = soup.find("p", class_="title-sub title-sub-2")
                row[header_map['8']] = varientClean(variant_tag.get_text(strip=True) if variant_tag else "")
                row[header_map["41"]] = extract_doors(variant_tag.get_text(strip=True) if variant_tag else "")
                
                row[header_map['13']]=auctionDetails.get("center","")
                row[header_map['14']] = auctionDetails.get("date","")
                row[header_map['15']] = auctionDetails.get("time","")

                findElement = extract_details(html_content)
                mot =  findElement.get("MOT", "")
                Registered =  findElement.get("Registered", "")
                row[header_map['16']] = findElement.get("Registered", "")
                row[header_map['11']] = findElement.get("Fuel", "")
                row[header_map['25']] = findElement.get("Former Keepers", "")
                row[header_map['12']] = findElement.get("Transmission", "")
                row[header_map['33']] = findElement.get("Colour", "")
                row[header_map['21']] = mot if mot != "Expired" else " "
                row[header_map["17"]] = yearGeter(Registered)
                row[header_map['34']] = findElement.get("Keys", "")
                row[header_map['23']] = findElement.get("VAT", "")
                row[header_map['20']] = findElement.get("V5", "")
                row[header_map['35']] = findElement.get("CAP Clean", "")
                row[header_map['36']] = findElement.get("CAP Below", "")
                row[header_map['37']] = findElement.get("CAP Average", "")
                row[header_map['46']] = findElement.get("Last service miles", "")

                cc_raw = findElement.get("CC", "")
                try:
                    cc_val = round(int(cc_raw) / 1000, 1) if cc_raw else ""
                except:
                    cc_val = ""
                row[header_map['26']] = cc_val


                warranted_raw = findElement.get("Miles Warranted", "")
                if warranted_raw.strip().lower() == "not warranted":
                    row[header_map['19']] = "NO"
                elif warranted_raw:
                    row[header_map['19']] = "Warranted"
                else:
                    row[header_map['19']] = ""
    
                miles_raw = findElement.get("Miles", "")
                miles_val = ""
                if miles_raw:
                    miles_clean = miles_raw.replace(",", "").replace("Miles", "").strip()

                    if miles_clean.replace(".", "").isdigit():  
                        miles_val = int(float(miles_clean)) 
                    else:
                        miles_val = ""  
                row[header_map['18']] = miles_val
        
                extractAdditional = extract_additional_info(html_content)
                info = extractAdditional.get("information", {})
                motDue = info.get("MOT Due", "")
                row[header_map['43']] = motDue if motDue != "Expired" else " "
                row[header_map['10']] = info.get("Body Style", "")
                row[header_map['22']] = info.get("Service history", "")
                row[header_map['45']] = info.get("Number of stamps", "")
                row[header_map['32']] = extractAdditional
                
                
                Grade = ""
                grade_span = soup.find("div", class_=lambda x: x and "nama-grade-" in x)

                if grade_span:
                    classes = grade_span.get("class", [])
                    for cls in classes:
                        if cls.startswith("nama-grade-") and cls != "nama-grade":
                            Grade = cls.replace("nama-grade-", "")
                            break

                row[header_map["40"]] = Grade

                ul = soup.find("ul", class_="details-list")
                data = {}
                for li in ul.find_all("li"):
                    span = li.find("span")
                    strong = li.find("strong")
                    if span and strong:
                        data[span.get_text(strip=True)] = strong.get_text(strip=True)
                nested_json = {"Interior": data}


                gernal_condition = json.dumps(nested_json, indent=4)
                row[header_map['39']] = gernal_condition
                
                
                tyres_div = None
                for div in soup.find_all("div"):
                    strong_tag = div.find("strong")
                    if strong_tag and strong_tag.get_text(strip=True) == "Tyres":
                        tyres_div = div
                        break



                
                
                base_url = "https://www.ebca.co.uk/"

                inspection_div = soup.find("div", class_="header-cta")
                inspection_pdf = ""
                if inspection_div:
                    a_tag = inspection_div.find("a", href=True)
                    if a_tag:
                        href = a_tag['href'].strip()
                        if href:
                            if href.startswith("http"):
                                inspection_pdf = href
                            else:
                                inspection_pdf = base_url.rstrip("/") + "/" + href.lstrip("/")

                row[header_map['28']] = inspection_pdf
                
                images_div = soup.find("div", class_="ug-thumbs-strip-inner")
                images_urls = []

                if images_div:
                    for img_tag in images_div.find_all("img", src=True):
                        src = img_tag["src"].strip()

                        if src:
                            src = src.replace(
                                "https://dgaww6lqj3.execute-api.eu-west-1.amazonaws.com/prod/buckets/ebca-prod-data-s3-public/keys/motorvehicle/",
                                "https://ebca-prod-data-s3-public.s3.eu-west-1.amazonaws.com/motorvehicle/"
  
                            )

                            src = src.replace("/resized/", "/")
                            src = src.split("---")[0] + ".jpg"

                            images_urls.append(src)
                            
                row[header_map['29']] = ", ".join(images_urls)
                damage_div = soup.find("div", class_="condition-gallery")
                damage_images = []
                damage_details = []

                damage_div = soup.find("div", class_="condition-gallery")
                if damage_div:
                    for figure in damage_div.find_all("figure", class_="condition-image"):
                        img_tag = figure.find("img", src=True)
                        if img_tag:
                            src = img_tag["src"].strip()
                            if src.startswith("/"):
                                src = base_url + src
                            src = src.replace("---1140-855.jpg", "---1024-768.jpg")
                            damage_images.append(src)

                        figcaption = figure.find("figcaption")
                        if figcaption:
                            caption = " ".join(figcaption.get_text(" ", strip=True).split())
                            if caption:
                                damage_details.append(caption)


                slider_div = soup.find("div", class_="ug-slider-wrapper")
                if slider_div:
                    for img in slider_div.find_all("img", src=True):
                        src = img["src"].strip()
                        if src and src not in damage_images:
                            damage_images.append(src)


                for txt in soup.find_all("div", class_="ug-textpanel-description"):
                    caption = " ".join(txt.get_text(" ", strip=True).split())
                    if caption and caption not in damage_details:
                        damage_details.append(caption)

                row[header_map['30']] = ", ".join(damage_images)
                row[header_map['31']] = ", ".join(damage_details)


                sheet_id, auction_name = get_base_folder_info()
                row[header_map['2']] = sheet_id or ""
                row[header_map['1']] = auction_name or ""
                row[header_map['3']] =  "East Anglian Motor Auctions"
                
                
            data = {}

            main_div = soup.find("div", class_="col-12 col-lg-5")
            if main_div:
                section_title = main_div.find("h3").get_text(strip=True)
                data[section_title] = {}

                for block in main_div.find_all("div", recursive=False):
                    block_title = block.find("strong").get_text(strip=True)

                    table = block.find("table")
                    if not table:
                        continue

                    block_data = {}

                    for tr in table.find_all("tr"):
                        tds = [td.get_text(" ", strip=True) for td in tr.find_all("td")]

                        if len(tds) == 2:
                            block_data[tds[0]] = tds[1]

                        elif len(tds) == 3:
                            block_data[tds[0]] = f"{tds[1]}\t{tds[2]}"

                    data[block_title] = block_data
            row[header_map['39']] = data
            tyres_data = data.get("Tyres", {})

            row[header_map['47']] = ", ".join(
                f"{k}: {v}" for k, v in tyres_data.items() if v
            )

            all_rows.append(row)
           


    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n✔ CSV Generated: {output_file}")


extract_manual_keys()


❌ Not valid: PGZ5491 → Deleting file PGZ5491.html

✔ CSV Generated: Ecba_data.csv


In [ ]:
import os
import threading
import requests
import pandas as pd
from urllib.parse import urlparse, urljoin
from PIL import Image, ImageDraw, ImageFont



def add_watermark_to_image(image_path, text="Sourced from Eastbourne Car Auctions"):
    try:
        image = Image.open(image_path).convert("RGBA")
        txt_layer = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(txt_layer)

        try:
            font = ImageFont.truetype("arial.ttf", 40)
        except:
            font = ImageFont.load_default()

        margin = 10
        bbox = draw.textbbox((0, 0), text, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]

        x = image.width - text_w - margin
        y = image.height - text_h - margin

        draw.rectangle(
            [x - margin, y - margin, x + text_w + margin, y + text_h + margin],
            fill=(0, 0, 0, 160)
        )
        draw.text((x, y), text, font=font, fill=(255, 255, 255, 255))

        final = Image.alpha_composite(image, txt_layer).convert("RGB")
        final.save(image_path)

        print(f"Watermark added: {image_path}")
    except Exception as e:
        print(f"Watermark fail {image_path}: {e}")


df = pd.read_csv("Ecba_data.csv")
reg_img = df[['Reg', 'Images']]
cond_imgs = df[['Damage_details', 'Damaged_images', 'Reg']]



def download_images(data, main_folder="Images"):
    os.makedirs(main_folder, exist_ok=True)

    for _, row in data.iterrows():
        reg = row["Reg"]
        images = row["Images"]

        if pd.isna(images) or not str(images).strip():
            print(f"No images for {reg}")
            continue

        urls = images.split(", ")
        reg_folder = os.path.join(main_folder, reg)
        os.makedirs(reg_folder, exist_ok=True)

        for idx, url in enumerate(urls):
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            try:
                response = requests.get(url, stream=True)
                response.raise_for_status()

                ext = os.path.splitext(urlparse(url).path)[1] or ".jpg"
                save_path = os.path.join(reg_folder, f"{reg}_{idx+1}{ext}")

                with open(save_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(save_path)
                print(f"Downloaded: {save_path}")

            except Exception as e:
                print(f"Image failed {url}: {e}")


def download_images_damage(data, main_folder="Damage_203"):
    os.makedirs(main_folder, exist_ok=True)

    for _, row in data.iterrows():
        reg = row["Reg"]
        dmg_imgs = row["Damaged_images"]
        dmg_texts = row["Damage_details"]

        if pd.isna(dmg_imgs) or pd.isna(dmg_texts):
            print(f"No damaged images for {reg}")
            continue

        urls = dmg_imgs.split(", ")
        texts = dmg_texts.split(", ")

        reg_folder = os.path.join(main_folder, reg)
        os.makedirs(reg_folder, exist_ok=True)

        for idx, (url, text) in enumerate(zip(urls, texts)):
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            safe_text = "".join(c if c.isalnum() or c in " _-" else "_" for c in text)

            try:
                response = requests.get(url, stream=True)
                response.raise_for_status()

                ext = os.path.splitext(urlparse(url).path)[1] or ".jpg"
                save_path = os.path.join(reg_folder, f"{safe_text}_{idx+1}{ext}")

                with open(save_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(save_path)
                print(f"Damage downloaded: {save_path}")

            except Exception as e:
                print(f"Damage failed {url}: {e}")



def start_funcs():
    threads = [
        threading.Thread(target=download_images, args=(reg_img,)),
        threading.Thread(target=download_images_damage, args=(cond_imgs,))
    ]

    for t in threads:
        t.start()

    for t in threads:
        t.join()


if __name__ == "__main__":
    start_funcs()


Watermark added: Images\AE70WRL\AE70WRL_1.jpg
Downloaded: Images\AE70WRL\AE70WRL_1.jpg
Watermark added: Damage_203\AE70WRL\Bonnet Chipped 1 To 5_1.jpg
Damage downloaded: Damage_203\AE70WRL\Bonnet Chipped 1 To 5_1.jpg
Watermark added: Damage_203\AF71CZD\Bonnet Chipped 1 To 5_1.jpg
Damage downloaded: Damage_203\AF71CZD\Bonnet Chipped 1 To 5_1.jpg
Watermark added: Damage_203\BD70OHK\Bonnet Scratched Over 25mm Thru Paint_1.jpg
Damage downloaded: Damage_203\BD70OHK\Bonnet Scratched Over 25mm Thru Paint_1.jpg
Watermark added: Damage_203\BG19SOA\Bonnet Chipped 1 To 5_1.jpg
Damage downloaded: Damage_203\BG19SOA\Bonnet Chipped 1 To 5_1.jpg
Watermark added: Damage_203\BP72TXK\Bonnet Chipped 1 To 5_1.jpg
Damage downloaded: Damage_203\BP72TXK\Bonnet Chipped 1 To 5_1.jpg
Watermark added: Damage_203\BP73POV\Bonnet Chipped 1 To 5_1.jpg
Damage downloaded: Damage_203\BP73POV\Bonnet Chipped 1 To 5_1.jpg
Watermark added: Damage_203\BT69XYS\Bonnet Chipped Multiple Chips_1.jpg
Damage downloaded: Damage_203

In [ ]:
import os
import time
import pandas as pd
import requests
import json

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


EMAIL = "fourbrotherstrading@icloud.com"
PASSWORD = "Muhssan7865@"


def setup_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver


def login(driver):
    driver.get("https://www.ebca.co.uk/login")
    wait = WebDriverWait(driver, 10)

    wait.until(EC.presence_of_element_located((By.ID, "username"))).send_keys(EMAIL)
    driver.find_element(By.ID, "password").send_keys(PASSWORD)
    driver.find_element(By.ID, "sign-in").click()

    time.sleep(2)
    print("[+] Login successful!")



def get_session_cookies(driver):
    cookies = driver.get_cookies()
    session = requests.Session()

    for cookie in cookies:
        session.cookies.set(cookie['name'], cookie['value'])

    return session


def download_pdf(session, url, save_path):
    print(f"[+] Downloading: {url}")

    response = session.get(url)
    if response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(response.content)

        print(f"[✔] Saved: {save_path}")
    else:
        print(f"[X] Failed ({response.status_code}) : {url}")


def download_all_pdfs(csv_file):
    df = pd.read_csv(csv_file)

    base_folder = "Inspection Reports"
    os.makedirs(base_folder, exist_ok=True)

    driver = setup_driver()
    login(driver)

    session = get_session_cookies(driver) 
    driver.quit()

    for _, row in df.iterrows():
        reg = row["Reg"]
        url = row["Inspection Report"]

        if pd.isna(url) or not str(url).strip():
            print(f"[!] Missing PDF for {reg}")
            continue

        save_path = os.path.join(base_folder, f"{reg}.pdf")
        download_pdf(session, url, save_path)

    print("\nAll PDFs downloaded!")



if __name__ == "__main__":
    download_all_pdfs("Ecba_data.csv")


[+] Login successful!
[+] Downloading: https://www.ebca.co.uk/motorvehicleinspectionreport/standard/6735.pdf
[✔] Saved: Inspection Reports\AO04KJX.pdf
[+] Downloading: https://www.ebca.co.uk/motorvehicleinspectionreport/standard/6737.pdf
[✔] Saved: Inspection Reports\AU64WOR.pdf
[+] Downloading: https://www.ebca.co.uk/motorvehicleinspectionreport/standard/6539.pdf
[✔] Saved: Inspection Reports\BF07CJZ.pdf
[+] Downloading: https://www.ebca.co.uk/motorvehicleinspectionreport/standard/6701.pdf
[✔] Saved: Inspection Reports\BG15MLN.pdf
[+] Downloading: https://www.ebca.co.uk/motorvehicleinspectionreport/standard/6261.pdf
[✔] Saved: Inspection Reports\BG19KFD.pdf
[+] Downloading: https://www.ebca.co.uk/motorvehicleinspectionreport/standard/6883.pdf
[✔] Saved: Inspection Reports\BG63VRY.pdf
[+] Downloading: https://www.ebca.co.uk/motorvehicleinspectionreport/standard/6800.pdf
[✔] Saved: Inspection Reports\BJ62ZFB.pdf
[+] Downloading: https://www.ebca.co.uk/motorvehicleinspectionreport/standa

In [ ]:
import os
from PyPDF2 import PdfReader, PdfWriter, Transformation
from reportlab.pdfgen import canvas

HEADER_HEIGHT = 25  
HEADER_WIDTH_PDF_GENRATOR = 595

def create_header_page(text, filename, page_width, page_height):
    c = canvas.Canvas(filename, pagesize=(page_width, page_height))
    r, g, b = (4/255, 122/255, 250/255)
    c.setFillColorRGB(r, g, b)
    c.rect(0, page_height - HEADER_HEIGHT, page_width, HEADER_HEIGHT, fill=1)
    c.setFillColorRGB(1, 1, 1)
    c.setFont("Helvetica-Bold", 12)

    text_width = c.stringWidth(text, "Helvetica-Bold", 12)
    x = (page_width - text_width) / 2
    y = page_height - HEADER_HEIGHT + 7

    c.drawString(x, y, text)
    c.save()


def testing(a):
    if not a:
        return ""
    v= a
    a = re.sub(r"^\s*\d+(\)")

def add_header_to_pdf(input_pdf, output_pdf):
    reader = PdfReader(input_pdf)
    writer = PdfWriter()

    for page in reader.pages:
        page_width = float(page.mediabox.width)
        page_height = float(page.mediabox.height)

        shift = Transformation().translate(0, -HEADER_HEIGHT)
        page.add_transformation(shift)

        temp_header = "header_temp.pdf"
        create_header_page("Source from Eastbourne Car Auctions", temp_header, page_width, page_height)

        header_pdf = PdfReader(temp_header)
        header_page = header_pdf.pages[0]

        page.merge_page(header_page)
        writer.add_page(page)

    with open(output_pdf, "wb") as f:
        writer.write(f)

    os.remove(temp_header)


def add_header_to_all_pdfs(folder):
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            input_pdf = os.path.join(folder, file)
            output_pdf = os.path.join(folder, file)

            print(f"Fixing & adding new header to: {file}")
            add_header_to_pdf(input_pdf, output_pdf)

    print("\n✔ All PDFs updated with blue header (#047AFA)!")


if __name__ == "__main__":
    add_header_to_all_pdfs("Inspection Reports")


Fixing & adding new header to: AO04KJX.pdf
Fixing & adding new header to: AU64WOR.pdf
Fixing & adding new header to: BF07CJZ.pdf
Fixing & adding new header to: BG15MLN.pdf
Fixing & adding new header to: BG19KFD.pdf
Fixing & adding new header to: BG63VRY.pdf
Fixing & adding new header to: BJ62ZFB.pdf
Fixing & adding new header to: BJ63WKV.pdf
Fixing & adding new header to: BL64CGV.pdf

✔ All PDFs updated with blue header (#047AFA)!
